d# Haja Coracao - Pipeline BPM sem otimizacoes




A versao otimizada esta em tratamento_batimentos_otimizado.ipynb para comparacao de planos, stages e DAG.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, stddev, min, max, count, sum as spark_sum, when, round as spark_round
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, BooleanType
import pandas as pd
import os
import time

spark = (SparkSession.builder
    .appName("HajaCoracao-BPM-Baseline")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "6")
    .getOrCreate())

spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Broadcast automatico:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))
print("Spark UI: http://localhost:4040")

In [ ]:
XLSX = "/home/ubuntu/dados_batimentos.xlsx"
CSV = "/home/ubuntu/dados_batimentos_baseline.csv"

if not os.path.exists(CSV):
    pd.read_excel(XLSX, sheet_name="Dados Batimentos").to_csv(CSV, index=False)

schema = StructType([
    StructField("messageId", IntegerType(), False),
    StructField("deviceId", StringType(), False),
    StructField("heartRate", DoubleType(), False),
    StructField("heartRateTarget", DoubleType(), False),
    StructField("activityState", IntegerType(), False),
    StructField("activityLabel", StringType(), False),
    StructField("bpmAlert", BooleanType(), False),
    StructField("timestamp", StringType(), False),
    StructField("deviceIndex", IntegerType(), False),
    StructField("timeSinceStart", StringType(), False)
])

bpm = spark.read.option("header", True).schema(schema).csv(CSV)
bpm.printSchema()
bpm.show(5, truncate=False)

In [ ]:
inicio = time.time()

dim_devices = spark.createDataFrame([
    ("device-01",), ("device-02",), ("device-03",),
    ("device-04",), ("device-05",), ("device-06",)
], ["deviceId"])

# Baseline: join comum, sem broadcast; o filtro ocorre depois do join.
bpm_completo = bpm.join(
    dim_devices,
    on="deviceId",
    how="inner"
 )

bpm_sem_otimizacao = (bpm_completo
    .filter(col("heartRate").isNotNull())
    .filter((col("heartRate") >= 30) & (col("heartRate") <= 220))
    .withColumn("timestamp", col("timestamp").cast("timestamp"))
 )

resultado_baseline = (bpm_sem_otimizacao
    .groupBy("deviceId", "activityLabel")
    .agg(
        count("*").alias("total_registros"),
        spark_round(avg("heartRate"), 2).alias("media_bpm"),
        spark_round(stddev("heartRate"), 2).alias("desvio_bpm"),
        min("heartRate").alias("min_bpm"),
        max("heartRate").alias("max_bpm"),
        spark_sum(when(col("bpmAlert") == True, 1).otherwise(0)).alias("alertas")
    )
 )

print("=== OPERACOES WIDE DA BASELINE ===")
print("join: pode gerar Exchange para redistribuir deviceId.")
print("groupBy: gera Exchange para reunir as chaves da agregacao.")
print("orderBy: gera Exchange adicional para a acao final comparavel.")
print("BroadcastExchange: nao esperado; broadcast automatico esta desligado.")
print("distinct/repartition/coalesce: nao usados.")
print("Filtro: aplicado depois do join.")
print("\n=== PLANO FISICO ===")
resultado_baseline.explain("formatted")
resultado_baseline.orderBy("deviceId", "activityLabel").show(20, truncate=False)

fim = time.time()
print(f"Tempo total de execucao: {fim - inicio:.2f} segundos")

In [ ]:
OUTPUT = "/home/ubuntu/processed_bpm_baseline_parquet"
resultado_baseline.write.mode("overwrite").parquet(OUTPUT)
print("Saida:", OUTPUT)
print("Use Jobs -> DAG Visualization na Spark UI para comparar com a versao otimizada.")